# Spatiotemporal Taxi Demand V4 XGBoost - Feature Engineering GCP



In [4]:
from functools import reduce

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans

BASE_HDFS = "/user/hieunh"
RAW_TRAIN = "/user/data/train/*.parquet"
RAW_VAL   = "/user/data/val/*.parquet"
RAW_TEST  = "/user/data/test/*.parquet"
FEATURE_ROOT = f"{BASE_HDFS}/spatiotemporal_xgboost_v4/features"

PICKUP_COUNTS_PATH = f"{FEATURE_ROOT}/pickup_counts_30m"
ZONE_TS_PATH = f"{FEATURE_ROOT}/zone_ts_matrix_30m"
CLUSTER_TS_PATH = f"{FEATURE_ROOT}/cluster_ts_matrix_30m"
CLUSTER_MAP_PATH = f"{FEATURE_ROOT}/cluster_map"
CORR_PATH = f"{FEATURE_ROOT}/corr_matrix_train"
DIST_PATH = f"{FEATURE_ROOT}/dist_matrix_train"
KERNEL_PATH = f"{FEATURE_ROOT}/kernel_matrix_train"
EMBEDDING_PATH = f"{FEATURE_ROOT}/embedding_train"
METADATA_PATH = f"{FEATURE_ROOT}/metadata"

SLOT_MINUTES = 30
MIN_MEAN_PICKUPS_PER_SLOT = 10.0
N_CLUSTERS = 8
CLUSTER_LAG = 12
DISAGG_LAG = 12
RANDOM_STATE = 42
PICKUP_COL_CANDIDATES = ["tpep_pickup_datetime", "pickup_datetime"]
ZONE_COL = "PULocationID"

spark = (
    SparkSession.builder
    .appName("Spatiotemporal_XGBoost_V4_FeatureEngineering_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.executor.instances", "2")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.memoryOverhead", "2g")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "12")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [ ]:
def load_split(path_glob: str, split_name: str):
    sc = spark.sparkContext
    fs = sc._jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
    file_status = fs.globStatus(sc._jvm.org.apache.hadoop.fs.Path(path_glob))
    if not file_status:
        raise ValueError(f"No files found for {path_glob}")

    dfs = []
    for status in file_status:
        p = status.getPath().toString()
        try:
            pickup_col_found = None
            df_tmp = spark.read.parquet(p)
            pickup_col_found = next(
                (c for c in PICKUP_COL_CANDIDATES if c in df_tmp.columns), None
            )
            if pickup_col_found is None or ZONE_COL not in df_tmp.columns:
                print(f"Skipping {p}: missing columns")
                continue

            # Select 2 cột 
            cleaned = (
                df_tmp.select(
                    F.col(pickup_col_found).cast("timestamp").alias("tpep_pickup_datetime"),
                    F.col(ZONE_COL).cast("int").alias(ZONE_COL),
                )
                .where(F.col("tpep_pickup_datetime").isNotNull())
                .where(F.col(ZONE_COL).isNotNull())
                .where(F.col(ZONE_COL) > 0)
                .where(F.col("tpep_pickup_datetime") >= F.lit("2020-01-01 00:00:00"))
                .where(F.col("tpep_pickup_datetime") < F.lit("2026-01-01 00:00:00"))
                .where(~(
                    (F.year("tpep_pickup_datetime") == 2020) &
                    (F.month("tpep_pickup_datetime").isin([3, 4, 5, 6]))
                ))
                .withColumn("split", F.lit(split_name))
            )
            dfs.append(cleaned)
            print("OK:", p)
        except Exception as e:
            print(f"Skipping {p}: {e}")
            continue

    if not dfs:
        raise ValueError(f"No valid data for {split_name}")
    return reduce(lambda a, b: a.unionByName(b), dfs)


def preprocess_pickup_counts_spark(df, slot_minutes=30):
    slot_seconds = slot_minutes * 60
    return (
        df.withColumn(
            "slot_unix",
            (F.floor(F.unix_timestamp("tpep_pickup_datetime") / slot_seconds) * slot_seconds).cast("long")
        )
        .withColumn("slot_ts", F.to_timestamp(F.from_unixtime(F.col("slot_unix"))))
        .groupBy("split", "slot_ts", ZONE_COL)
        .agg(F.count(F.lit(1)).alias("pickup_count"))
        .withColumnRenamed(ZONE_COL, "zone_id")
        .select("split", "slot_ts", "zone_id", "pickup_count")
    )


def get_active_zones_spark(pickup_counts_sdf, min_mean_pickups_per_slot=10.0):
    train_counts = pickup_counts_sdf.where(F.col("split") == "train")
    n_slots = train_counts.select("slot_ts").distinct().count()
    zone_stats = (
        train_counts
        .groupBy("zone_id")
        .agg(F.sum("pickup_count").alias("total_pickups"))
        .withColumn("mean_pickups_per_slot", F.col("total_pickups") / F.lit(n_slots))
        .filter(F.col("mean_pickups_per_slot") >= F.lit(min_mean_pickups_per_slot))
        .orderBy("zone_id")
    )
    return [r["zone_id"] for r in zone_stats.select("zone_id").collect()]


def build_dense_zone_slot_matrix_spark(pickup_counts_sdf, active_zones, split_name, slot_minutes=30):
    sdf = pickup_counts_sdf.filter((F.col("split") == split_name) & F.col("zone_id").isin(active_zones))
    min_max = sdf.select(F.min("slot_ts").alias("min_ts"), F.max("slot_ts").alias("max_ts")).collect()[0]
    min_ts = min_max["min_ts"]
    max_ts = min_max["max_ts"]
    if min_ts is None or max_ts is None:
        raise ValueError(f"No rows for split={split_name}")

    full_slots = spark.sql(f"""
        SELECT explode(
            sequence(
                timestamp('{min_ts}'),
                timestamp('{max_ts}'),
                interval {slot_minutes} minutes
            )
        ) AS slot_ts
    """)
    zones_df = spark.createDataFrame([(z,) for z in active_zones], ["zone_id"])
    dense_base = full_slots.crossJoin(zones_df)
    dense_counts = dense_base.join(sdf, on=["slot_ts", "zone_id"], how="left").fillna(0, subset=["pickup_count"])
    pivot_sdf = (
        dense_counts
        .groupBy("slot_ts")
        .pivot("zone_id", active_zones)
        .sum("pickup_count")
        .na.fill(0)
        .orderBy("slot_ts")
    )
    pdf = pivot_sdf.toPandas()
    pdf["slot_ts"] = pd.to_datetime(pdf["slot_ts"])
    pdf = pdf.set_index("slot_ts")
    pdf.columns = [int(c) for c in pdf.columns]
    pdf = pdf.sort_index(axis=1)
    return pdf.astype(np.float32)


def compute_correlation_dissimilarity(zone_ts_pdf: pd.DataFrame):
    corr_df = zone_ts_pdf.corr(method="pearson").fillna(0.0).clip(-1.0, 1.0)
    corr_arr = corr_df.values.copy()  # ← thêm .copy()
    np.fill_diagonal(corr_arr, 1.0)
    corr_df = pd.DataFrame(corr_arr, index=corr_df.index, columns=corr_df.columns)
    return corr_df, 1.0 - corr_df


def gaussian_kernel_on_distance_rows(dist_df: pd.DataFrame):
    D = dist_df.values.astype(np.float64)
    L = D.shape[0]
    row_sq = np.sum(D * D, axis=1, keepdims=True)
    sq_euclidean = row_sq + row_sq.T - 2 * (D @ D.T)
    sq_euclidean = np.maximum(sq_euclidean, 0.0)
    G = np.exp(-sq_euclidean / max(L, 1))
    return pd.DataFrame(G, index=dist_df.index, columns=dist_df.columns)


def cluster_zones_kernel_pca_kmeans(dist_df: pd.DataFrame, n_clusters: int = 8, random_state: int = 42):
    G_df = gaussian_kernel_on_distance_rows(dist_df)
    n_components = min(max(n_clusters, 2), max(1, G_df.shape[0] - 1))
    kpca = KernelPCA(n_components=n_components, kernel="precomputed", random_state=random_state)
    embedding = kpca.fit_transform(G_df.values)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
    labels = kmeans.fit_predict(embedding)
    cluster_map = pd.Series(labels, index=dist_df.index, name="cluster_id")
    return cluster_map, G_df, embedding


def aggregate_cluster_demand(zone_ts_pdf: pd.DataFrame, cluster_map: pd.Series):
    cluster_ts = {}
    for c in sorted(cluster_map.unique()):
        zones = cluster_map[cluster_map == c].index.tolist()
        cluster_ts[c] = zone_ts_pdf[zones].sum(axis=1)
    cluster_ts_pdf = pd.DataFrame(cluster_ts, index=zone_ts_pdf.index).sort_index(axis=1)
    cluster_ts_pdf.columns = [f"cluster_{c}" for c in cluster_ts_pdf.columns]
    return cluster_ts_pdf


def matrix_to_spark(pdf: pd.DataFrame, split_name: str, index_name="slot_ts"):
    out = pdf.reset_index().copy()
    out.insert(0, "split", split_name)
    out[index_name] = pd.to_datetime(out[index_name])
    out.columns = [str(c) if c not in ["split", index_name] else c for c in out.columns]
    return spark.createDataFrame(out)


def square_to_spark(pdf: pd.DataFrame, index_name="zone_id"):
    out = pdf.reset_index().copy()
    first_col = out.columns[0]
    out = out.rename(columns={first_col: index_name})
    out.columns = [str(c) if c != index_name else index_name for c in out.columns]
    return spark.createDataFrame(out)


In [6]:
print("Step 1/6: read train/val/test from HDFS...")
train_df = load_split(RAW_TRAIN, "train")
val_df = load_split(RAW_VAL, "val")
test_df = load_split(RAW_TEST, "test")
raw_df = train_df.unionByName(val_df).unionByName(test_df).cache()
print("Raw rows:", raw_df.count())
raw_df.groupBy("split").count().show()
raw_df.groupBy("split").agg(F.min("tpep_pickup_datetime").alias("min_time"), F.max("tpep_pickup_datetime").alias("max_time")).show(truncate=False)

print("Step 2/6: preprocess pickup counts in Spark...")
pickup_counts_sdf = preprocess_pickup_counts_spark(raw_df, slot_minutes=SLOT_MINUTES).persist()
print("pickup_counts rows:", pickup_counts_sdf.count())
pickup_counts_sdf.groupBy("split").agg(F.countDistinct("zone_id").alias("zones"), F.countDistinct("slot_ts").alias("slots")).show()

print("Step 3/6: find active zones from TRAIN only...")
active_zones = get_active_zones_spark(pickup_counts_sdf, MIN_MEAN_PICKUPS_PER_SLOT)
print("Number of active zones:", len(active_zones))
print("Some active zones:", active_zones[:10])

print("Step 4/6: build dense zone-slot matrices for train/val/test...")
zone_ts_by_split = {
    split_name: build_dense_zone_slot_matrix_spark(pickup_counts_sdf, active_zones, split_name, SLOT_MINUTES)
    for split_name in ["train", "val", "test"]
}
for split_name, pdf in zone_ts_by_split.items():
    print(split_name, pdf.shape, pdf.index.min(), pdf.index.max())

print("Step 5/6: compute train correlation/dissimilarity and cluster zones...")
corr_df, dist_df = compute_correlation_dissimilarity(zone_ts_by_split["train"])
cluster_map, kernel_df, embedding = cluster_zones_kernel_pca_kmeans(dist_df, N_CLUSTERS, RANDOM_STATE)
print(cluster_map.value_counts().sort_index())
print(f"Total zones in clusters: {len(cluster_map)}")

cluster_ts_by_split = {
    split_name: aggregate_cluster_demand(pdf, cluster_map)
    for split_name, pdf in zone_ts_by_split.items()
}
for split_name, pdf in cluster_ts_by_split.items():
    print(split_name, pdf.shape)

print("Step 6/6: save artifacts to HDFS...")
pickup_counts_sdf.write.mode("overwrite").parquet(PICKUP_COUNTS_PATH)
reduce(lambda a, b: a.unionByName(b), [matrix_to_spark(pdf, split_name) for split_name, pdf in zone_ts_by_split.items()]).write.mode("overwrite").partitionBy("split").parquet(ZONE_TS_PATH)
reduce(lambda a, b: a.unionByName(b), [matrix_to_spark(pdf, split_name) for split_name, pdf in cluster_ts_by_split.items()]).write.mode("overwrite").partitionBy("split").parquet(CLUSTER_TS_PATH)
square_to_spark(corr_df).write.mode("overwrite").parquet(CORR_PATH)
square_to_spark(dist_df).write.mode("overwrite").parquet(DIST_PATH)
square_to_spark(kernel_df).write.mode("overwrite").parquet(KERNEL_PATH)

cluster_map_df = pd.DataFrame({"zone_id": cluster_map.index.astype(int), "cluster_id": cluster_map.values.astype(int)})
spark.createDataFrame(cluster_map_df).write.mode("overwrite").parquet(CLUSTER_MAP_PATH)

embedding_cols = [f"kpca_{i}" for i in range(embedding.shape[1])]
embedding_df = pd.DataFrame(embedding, columns=embedding_cols)
embedding_df.insert(0, "zone_id", cluster_map.index.astype(int))
spark.createDataFrame(embedding_df).write.mode("overwrite").parquet(EMBEDDING_PATH)

metadata_rows = [
    ("slot_minutes", str(SLOT_MINUTES)),
    ("min_mean_pickups_per_slot", str(MIN_MEAN_PICKUPS_PER_SLOT)),
    ("n_clusters", str(N_CLUSTERS)),
    ("cluster_lag", str(CLUSTER_LAG)),
    ("disagg_lag", str(DISAGG_LAG)),
    ("random_state", str(RANDOM_STATE)),
    ("n_active_zones", str(len(active_zones))),
]
spark.createDataFrame(metadata_rows, ["key", "value"]).write.mode("overwrite").parquet(METADATA_PATH)

print("Saved HDFS outputs:")
for p in [PICKUP_COUNTS_PATH, ZONE_TS_PATH, CLUSTER_TS_PATH, CLUSTER_MAP_PATH, CORR_PATH, DIST_PATH, KERNEL_PATH, EMBEDDING_PATH, METADATA_PATH]:
    print("-", p)


Step 1/6: read train/val/test from HDFS...
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-01.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-02.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-03.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-04.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-05.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-06.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-07.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-08.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-09.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-10.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-11.parquet
OK: hdfs://hadoop-master:9000/user/data/train/yellow_tripdata_2020-12.parquet
OK: hdfs://hadoop-mas

26/05/28 01:50:14 WARN CacheManager: Asked to cache already cached data.


Raw rows: 219267313


+-----+---------+
|split|    count|
+-----+---------+
|train|135347183|
|  val| 24049032|
| test| 59871098|
+-----+---------+



26/05/28 01:50:49 WARN CacheManager: Asked to cache already cached data.        


...

Number of active zones: 46
Some active zones: [13, 43, 48, 50, 68, 75, 79, 87, 90, 100]
Step 4/6: build dense zone-slot matrices for train/val/test...


26/05/28 01:51:00 WARN DAGScheduler: Broadcasting large task binary with size 1275.1 KiB
26/05/28 01:51:04 WARN DAGScheduler: Broadcasting large task binary with size 1317.9 KiB
26/05/28 01:51:05 WARN DAGScheduler: Broadcasting large task binary with size 1317.9 KiB
26/05/28 01:51:06 WARN DAGScheduler: Broadcasting large task binary with size 1300.1 KiB


train (73009, 46) 2020-01-01 00:00:00 2024-03-01 00:00:00
val (10317, 46) 2024-02-29 23:00:00 2024-10-01 21:00:00
test (21937, 46) 2024-09-30 23:30:00 2025-12-31 23:30:00
Step 5/6: compute train correlation/dissimilarity and cluster zones...
cluster_id
0    13
1     3
2     3
3     8
4     1
5    12
6     2
7     4
Name: count, dtype: int64
Total zones in clusters: 46
train (73009, 8)
val (10317, 8)
test (21937, 8)
Step 6/6: save artifacts to HDFS...


26/05/28 01:51:38 WARN TaskSetManager: Stage 416 contains a task of very large size (5241 KiB). The maximum recommended task size is 1000 KiB.
26/05/28 01:51:46 WARN TaskSetManager: Stage 417 contains a task of very large size (1117 KiB). The maximum recommended task size is 1000 KiB.


Saved HDFS outputs:
- /user/hieunh/spatiotemporal_xgboost_v4/features/pickup_counts_30m
- /user/hieunh/spatiotemporal_xgboost_v4/features/zone_ts_matrix_30m
- /user/hieunh/spatiotemporal_xgboost_v4/features/cluster_ts_matrix_30m
- /user/hieunh/spatiotemporal_xgboost_v4/features/cluster_map
- /user/hieunh/spatiotemporal_xgboost_v4/features/corr_matrix_train
- /user/hieunh/spatiotemporal_xgboost_v4/features/dist_matrix_train
- /user/hieunh/spatiotemporal_xgboost_v4/features/kernel_matrix_train
- /user/hieunh/spatiotemporal_xgboost_v4/features/embedding_train
- /user/hieunh/spatiotemporal_xgboost_v4/features/metadata


In [7]:
spark.catalog.clearCache()
spark.stop()
